# Tipos de Ordenes y Matching Behavior

Del LOB como estructura al LOB en movimiento.

## Objetivo de la sesion

- entender como una market order consume niveles del libro
- construir un `MatchingEngine` minimo en Python con fills, slippage, y precio medio
- implementar limit orders, IOC y FOK

Antes de ejecutar cada celda, intenta predecir que deberia salir.

## 0. El puente desde Lesson 4

En L4 cargaste `btc_lob_snapshots.csv` y calculaste spread, mid price, imbalance y weighted mid. El LOB era una estructura estatica: una foto del libro en un instante.

Hoy vas a interactuar con ese mismo libro. Enviarle ordenes. Ver como reacciona. El mismo CSV, la misma fila 0 — pero ahora como punto de partida de un motor de matching.

**Antes de ejecutar:** la fila 0 tiene `bid_price_1 = 99978.20` y `ask_price_1 = 100021.80`. Si envias una market buy de 0.5 BTC, ¿a que precio ejecutas? ¿Es exactamente `ask_price_1`?

## 1. Cargar el LOB de L4

Reutilizamos los datos de la clase anterior. El snapshot 0 sera nuestro punto de partida.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import copy
import uuid

df = pd.read_csv("../04-market-microstructure-btc/data/btc_lob_snapshots.csv")
row = df.iloc[0]

# Convertir la fila en listas de dicts — formato que usara el motor
bids_raw = [{"price": row[f"bid_price_{i}"], "size": row[f"bid_size_{i}"], "id": f"b{i}"}
            for i in range(1, 11)]
asks_raw = [{"price": row[f"ask_price_{i}"], "size": row[f"ask_size_{i}"], "id": f"a{i}"}
            for i in range(1, 11)]

print("Snapshot 0 — primeros 3 niveles:")
print(f"  Best bid: ${bids_raw[0]['price']:,.2f} x {bids_raw[0]['size']} BTC")
print(f"  Best ask: ${asks_raw[0]['price']:,.2f} x {asks_raw[0]['size']} BTC")
print(f"  Spread:   ${asks_raw[0]['price'] - bids_raw[0]['price']:.2f}")
print(f"\nBids (top 3): {[(b['price'], b['size']) for b in bids_raw[:3]]}")
print(f"Asks (top 3): {[(a['price'], a['size']) for a in asks_raw[:3]]}")

## 2. El MatchingEngine

Vamos a encapsular la logica en una clase. El motor guarda el estado del libro (bids y asks) y expone metodos para interactuar con el.

**Antes de ejecutar:** el motor hace una copia profunda del libro para no modificar los datos originales. ¿Por que es importante eso?

In [ ]:
class MatchingEngine:
    """Motor de matching FIFO minimo para un snapshot del LOB."""

    def __init__(self, bids, asks):
        # Copia profunda: el motor modifica el libro, no queremos tocar los datos originales
        self.bids = copy.deepcopy(bids)
        self.asks = copy.deepcopy(asks)

    @classmethod
    def from_row(cls, row):
        """Construir desde una fila del DataFrame de L4."""
        bids = [{"price": row[f"bid_price_{i}"], "size": row[f"bid_size_{i}"], "id": f"b{i}"}
                for i in range(1, 11)]
        asks = [{"price": row[f"ask_price_{i}"], "size": row[f"ask_size_{i}"], "id": f"a{i}"}
                for i in range(1, 11)]
        return cls(bids, asks)

    def best_bid(self):
        return self.bids[0]["price"] if self.bids else None

    def best_ask(self):
        return self.asks[0]["price"] if self.asks else None

    def spread(self):
        if self.bids and self.asks:
            return self.asks[0]["price"] - self.bids[0]["price"]
        return None

    def __repr__(self):
        bb = f"${self.best_bid():,.2f}" if self.best_bid() else "—"
        ba = f"${self.best_ask():,.2f}" if self.best_ask() else "—"
        sp = f"${self.spread():.2f}" if self.spread() else "—"
        return f"MatchingEngine(bid={bb}, ask={ba}, spread={sp}, bids={len(self.bids)}, asks={len(self.asks)})"


engine = MatchingEngine.from_row(row)
print(repr(engine))

## 3. Market buy — caminar el ask side

Una market buy consume los niveles del ask en orden de precio (FIFO: el mejor precio primero, y dentro del mismo precio, por orden de llegada).

**Antes de ejecutar:** el best ask tiene `size = 1.5905 BTC`. Si envias una market buy de 0.5 BTC, ¿consume un solo nivel o varios?

In [ ]:
def market_buy(engine, size):
    """
    Envia una market buy al motor. Consume asks de menor a mayor precio.
    Devuelve: fills, avg_price, slippage, residual.
    """
    orig_ask = engine.best_ask()
    fills, remaining, cost = [], size, 0.0

    for level in engine.asks:
        if remaining <= 0:
            break
        filled = min(remaining, level["size"])
        fills.append({"price": level["price"], "size": round(filled, 6)})
        cost += filled * level["price"]
        level["size"] = round(level["size"] - filled, 6)
        remaining = round(remaining - filled, 6)

    # Eliminar niveles vacios
    engine.asks = [l for l in engine.asks if l["size"] > 0.0001]

    total_filled = round(size - remaining, 6)
    avg_price = cost / total_filled if total_filled > 0 else 0
    slippage = avg_price - orig_ask if avg_price > 0 else 0

    return {
        "fills": fills,
        "total_filled": total_filled,
        "avg_price": round(avg_price, 4),
        "slippage": round(slippage, 4),
        "residual": remaining,
    }


# Recrear el motor para este ejemplo
engine = MatchingEngine.from_row(row)

print("--- Market buy 0.5 BTC ---")
result = market_buy(engine, 0.5)
print(f"fills: {result['fills']}")
print(f"total ejecutado: {result['total_filled']} BTC")
print(f"precio medio:    ${result['avg_price']:,.4f}")
print(f"slippage:        ${result['slippage']:.4f}")
print(f"residual:        {result['residual']} BTC")
print(f"\nEstado del libro despues:")
print(repr(engine))

## 4. Visualizar los fills

Una market buy de 5 BTC va a consumir varios niveles. Vamos a visualizar exactamente cuales y cuanto de cada uno.

**Antes de ejecutar:** el best ask tiene 1.59 BTC. El segundo nivel tiene 0.68 BTC. El tercero 0.15 BTC. ¿Cuantos niveles consume una market buy de 5 BTC?

In [ ]:
engine = MatchingEngine.from_row(row)
original_asks = copy.deepcopy(engine.asks[:8])  # guardar estado antes

result_5 = market_buy(engine, 5.0)

fig, ax = plt.subplots(figsize=(10, 5))

# Todos los niveles originales en gris
prices_orig  = [l["price"] for l in original_asks]
sizes_orig   = [l["size"]  for l in original_asks]
ax.barh(prices_orig, sizes_orig, height=6, color="#3f3f46", alpha=0.6, label="Disponible original")

# Niveles consumidos por el fill en rojo
fill_by_price = {f["price"]: f["size"] for f in result_5["fills"]}
prices_fill = list(fill_by_price.keys())
sizes_fill  = list(fill_by_price.values())
ax.barh(prices_fill, sizes_fill, height=6, color="#f87171", alpha=0.9, label="Consumido (fill)")

# Precio medio de ejecucion
ax.axhline(y=result_5["avg_price"], color="#22d3ee", linestyle="--", linewidth=1.5,
           label=f"Precio medio ${result_5['avg_price']:,.2f}")

ax.set_xlabel("Volumen (BTC)")
ax.set_ylabel("Precio (USD)")
ax.set_title(f"Market Buy 5 BTC — {len(result_5['fills'])} niveles consumidos, slippage ${result_5['slippage']:.2f}")
ax.legend()
ax.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Fills detallados: {result_5['fills']}")
print(f"Slippage: ${result_5['slippage']:.4f} ({result_5['slippage']/asks_raw[0]['price']*100:.4f}%)")

## 5. Market sell — caminar el bid side

Simetrico al market buy: consume bids de mayor a menor precio.

**Antes de ejecutar:** el market sell cobra menos que `best_bid` si la orden es suficientemente grande. El slippage aqui es `best_bid - avg_price`. ¿Por que tiene el mismo mecanismo que el market buy pero en sentido contrario?

In [ ]:
def market_sell(engine, size):
    """
    Envia una market sell. Consume bids de mayor a menor precio.
    """
    orig_bid = engine.best_bid()
    fills, remaining, revenue = [], size, 0.0

    for level in engine.bids:
        if remaining <= 0:
            break
        filled = min(remaining, level["size"])
        fills.append({"price": level["price"], "size": round(filled, 6)})
        revenue += filled * level["price"]
        level["size"] = round(level["size"] - filled, 6)
        remaining = round(remaining - filled, 6)

    engine.bids = [l for l in engine.bids if l["size"] > 0.0001]

    total_filled = round(size - remaining, 6)
    avg_price = revenue / total_filled if total_filled > 0 else 0
    slippage = orig_bid - avg_price if avg_price > 0 else 0  # cuanto menos cobras respecto al best bid

    return {
        "fills": fills,
        "total_filled": total_filled,
        "avg_price": round(avg_price, 4),
        "slippage": round(slippage, 4),
        "residual": remaining,
    }


engine = MatchingEngine.from_row(row)

print("--- Market sell 0.5 BTC ---")
r = market_sell(engine, 0.5)
print(f"fills:         {r['fills']}")
print(f"precio medio:  ${r['avg_price']:,.4f}")
print(f"slippage:      ${r['slippage']:.4f}")

print("\n--- Market sell 5.0 BTC (cruza varios niveles) ---")
engine2 = MatchingEngine.from_row(row)
r2 = market_sell(engine2, 5.0)
print(f"niveles consumidos: {len(r2['fills'])}")
print(f"precio medio:       ${r2['avg_price']:,.4f}")
print(f"slippage:           ${r2['slippage']:.4f}")

## 6. Limit orders — colocar en el libro

Una limit order no ejecuta inmediatamente. Se coloca en el libro al precio indicado y espera contrapartida. Si el precio cruza el spread al colocarla, se ejecuta como market order.

**Antes de ejecutar:** si colocas una limit bid a $100.025 — por encima del best ask ($100.021) — ¿que deberia pasar?

In [ ]:
def place_limit(engine, side, price, size):
    """
    Coloca una limit order en el libro.
    Si cruza el spread (precio >= best_ask para bid, precio <= best_bid para ask),
    se ejecuta inmediatamente como market order.
    """
    order_id = str(uuid.uuid4())[:8]

    # Comprobacion de cruce de spread
    if side == "bid" and engine.asks and price >= engine.asks[0]["price"]:
        print(f"  Limit bid a ${price} cruza el spread (best ask ${engine.asks[0]['price']}) → ejecuta como market")
        return {**market_buy(engine, size), "type": "LIMIT-CROSS", "id": order_id}

    if side == "ask" and engine.bids and price <= engine.bids[0]["price"]:
        print(f"  Limit ask a ${price} cruza el spread (best bid ${engine.bids[0]['price']}) → ejecuta como market")
        return {**market_sell(engine, size), "type": "LIMIT-CROSS", "id": order_id}

    # Insertar en posicion correcta (mayor a menor para bids, menor a mayor para asks)
    new_level = {"price": price, "size": size, "id": order_id, "is_user": True}

    if side == "bid":
        idx = next((i for i, b in enumerate(engine.bids) if b["price"] <= price), len(engine.bids))
        engine.bids.insert(idx, new_level)
        status = f"Limit bid ${price} x {size} BTC → en cola (posicion {idx+1} del bid side)"
    else:
        idx = next((i for i, a in enumerate(engine.asks) if a["price"] >= price), len(engine.asks))
        engine.asks.insert(idx, new_level)
        status = f"Limit ask ${price} x {size} BTC → en cola (posicion {idx+1} del ask side)"

    return {"type": "LIMIT-RESTING", "id": order_id, "side": side, "price": price, "size": size, "status": status}


def cancel_limit(engine, order_id):
    """Cancela una limit order por su ID."""
    before = len(engine.bids) + len(engine.asks)
    engine.bids = [b for b in engine.bids if b.get("id") != order_id]
    engine.asks = [a for a in engine.asks if a.get("id") != order_id]
    after = len(engine.bids) + len(engine.asks)
    return {"cancelled": before > after, "id": order_id}


engine = MatchingEngine.from_row(row)
print(f"Estado inicial: {repr(engine)}")

# Caso 1: limit bid por debajo del best ask — queda en el libro
print("\n--- Limit bid a $99,960 (por debajo del spread) ---")
r1 = place_limit(engine, "bid", 99960.0, 0.5)
print(r1["status"])
print(f"Best bid ahora: ${engine.best_bid():,.2f} (nuestro limit NO es el mejor)")

# Caso 2: limit bid que cruza el spread
print("\n--- Limit bid a $100,025 (cruza el spread) ---")
engine2 = MatchingEngine.from_row(row)
r2 = place_limit(engine2, "bid", 100025.0, 0.3)
print(f"Fills: {r2['fills']}")

# Cancelar la orden del caso 1
print("\n--- Cancelar limit bid del caso 1 ---")
result_cancel = cancel_limit(engine, r1["id"])
print(f"Cancelada: {result_cancel['cancelled']}")

## 7. IOC — Immediate or Cancel

IOC ejecuta todo lo que puede ahora. Si queda residuo sin ejecutar, lo cancela. Nunca deja ordenes en el libro.

**Antes de ejecutar:** si el libro solo tiene 8 BTC disponibles en el ask side y envias un IOC de 10 BTC, ¿que fill esperas? ¿que residual?

In [ ]:
def ioc_buy(engine, size):
    """
    IOC buy: ejecuta lo que pueda, cancela el residuo.
    El residuo NO se anade al libro — se descarta.
    """
    result = market_buy(engine, size)
    # market_buy ya devuelve el residual sin anadir nada al libro
    return {**result, "type": "IOC", "cancelled": result["residual"]}


engine = MatchingEngine.from_row(row)

# IOC con tamano razonable — deberia ejecutar completamente
print("--- IOC buy 1.0 BTC ---")
r_ioc_small = ioc_buy(engine, 1.0)
print(f"  filled:    {r_ioc_small['total_filled']} BTC")
print(f"  cancelled: {r_ioc_small['cancelled']} BTC")
print(f"  avg_price: ${r_ioc_small['avg_price']:,.4f}")

# IOC con tamano grande — residuo se cancela
print("\n--- IOC buy 20 BTC (mas de lo disponible en el libro) ---")
engine2 = MatchingEngine.from_row(row)
total_ask_vol = sum(l["size"] for l in engine2.asks)
print(f"  Volumen total en asks: {total_ask_vol:.4f} BTC")

r_ioc_big = ioc_buy(engine2, 20.0)
print(f"  filled:    {r_ioc_big['total_filled']:.4f} BTC")
print(f"  cancelled: {r_ioc_big['cancelled']:.4f} BTC (residuo descartado, NO va al libro)")
print(f"\n  Motor despues: {repr(engine2)} — asks vacios")

## 8. FOK — Fill or Kill

FOK es mas estricto que IOC: primero verifica si hay liquidez suficiente para completar la orden entera. Si no la hay, cancela todo sin ejecutar nada.

**Antes de ejecutar:** ¿en que situacion preferirías un FOK sobre un IOC?

In [ ]:
def fok_buy(engine, size):
    """
    FOK buy: comprueba si hay liquidez suficiente ANTES de ejecutar.
    Si no hay suficiente, cancela toda la orden sin ejecutar nada.
    """
    available = sum(l["size"] for l in engine.asks)

    if available < size - 0.0001:
        return {
            "type": "FOK",
            "killed": True,
            "reason": f"liquidez insuficiente: disponible {available:.4f} BTC < requerido {size} BTC",
            "fills": [],
            "total_filled": 0,
        }

    # Hay liquidez — ejecutar como market buy
    result = market_buy(engine, size)
    return {**result, "type": "FOK", "killed": False}


engine = MatchingEngine.from_row(row)

# FOK que ejecuta
print("--- FOK buy 2.0 BTC ---")
r_fok_ok = fok_buy(engine, 2.0)
print(f"  killed:    {r_fok_ok['killed']}")
print(f"  filled:    {r_fok_ok['total_filled']} BTC")
print(f"  avg_price: ${r_fok_ok['avg_price']:,.4f}")

# FOK que se mata (20 BTC > liquidity del libro)
print("\n--- FOK buy 20.0 BTC ---")
engine2 = MatchingEngine.from_row(row)
r_fok_kill = fok_buy(engine2, 20.0)
print(f"  killed: {r_fok_kill['killed']}")
print(f"  razon:  {r_fok_kill['reason']}")
print(f"  Motor intacto: {repr(engine2)} — sin cambios")

## 9. Comparar coste: market vs limit

La limit order ahorra el spread — pero solo si ejecuta. Vamos a cuantificar esa diferencia.

**Antes de ejecutar:** si el spread es ~$43, ¿cuanto ahorras en un trade de 1 BTC eligiendo limit sobre market?

In [ ]:
engine_m = MatchingEngine.from_row(row)
engine_l = MatchingEngine.from_row(row)

size = 1.0
mid  = (engine_m.best_bid() + engine_m.best_ask()) / 2

# Market order: paga el ask
r_market = market_buy(engine_m, size)

# Limit order al mid: si el mercado baja hasta el mid, ejecuta a ese precio
# Simulamos: la "limit al mid" ejecutaria a mid (simplificacion pedagogica)
limit_price = round(mid, 2)
print(f"Mid price:         ${mid:,.2f}")
print(f"Best ask:          ${engine_l.best_ask():,.2f}")
print()
print(f"--- Market buy {size} BTC ---")
print(f"  Precio medio:    ${r_market['avg_price']:,.2f}")
print(f"  Coste vs mid:    ${r_market['avg_price'] - mid:.2f}")
print()
print(f"--- Limit buy al mid (${limit_price:,.2f}) ---")
print(f"  Precio si ejecuta: ${limit_price:,.2f}")
print(f"  Ahorro vs market:  ${r_market['avg_price'] - limit_price:.2f} por BTC")
print()

# Spread actual de este snapshot (es amplio por ser snapshot 0 con spread inusual)
spread = engine_l.best_ask() - engine_l.best_bid()
print(f"Spread actual:     ${spread:.2f} (nota: este snapshot tiene spread amplio)")
print(f"Spread medio L4:   $11.39")
print()
print("En condiciones normales (spread ~$11):")
print(f"  Ahorro limit vs market ≈ $5-6 por BTC (medio spread)")

## 10. Tu turno — `simulate_sequence`

Implementa una funcion `simulate_sequence(engine, orders)` que procese una lista de ordenes en orden y devuelva un resumen de cada ejecucion.

Cada orden es un dict con:
- `type`: `"market_buy"`, `"market_sell"`, `"limit_bid"`, `"limit_ask"`, `"ioc_buy"`, `"fok_buy"`
- `size`: tamano en BTC
- `price` (solo para limit): precio limite

Debe devolver una lista de resultados con el tipo de orden y el fill obtenido.

**Pista:** usa un `if/elif` sobre `order["type"]` y llama a la funcion correspondiente.

In [ ]:
def simulate_sequence(engine, orders):
    """
    Procesa una lista de ordenes en orden y devuelve un resumen de cada ejecucion.

    Parametros:
        engine: MatchingEngine — se modifica en cada orden
        orders: lista de dicts con claves 'type', 'size', y opcionalmente 'price'

    Retorna:
        lista de dicts con 'order_type', 'size', y el resultado de la ejecucion
    """
    results = []

    for order in orders:
        # Escribe aqui tu implementacion
        pass

    return results


# Prueba tu implementacion con esta secuencia:
engine_test = MatchingEngine.from_row(row)

test_orders = [
    {"type": "market_buy",  "size": 0.5},
    {"type": "limit_bid",   "size": 0.3, "price": 99950.0},
    {"type": "market_sell", "size": 0.5},
    {"type": "ioc_buy",     "size": 1.5},
    {"type": "fok_buy",     "size": 20.0},
]

seq_results = simulate_sequence(engine_test, test_orders)
for i, r in enumerate(seq_results):
    print(f"Orden {i+1}: {r}")

### Solucion — `simulate_sequence`

Despliega solo despues de intentarlo.

In [ ]:
def simulate_sequence(engine, orders):
    results = []
    for order in orders:
        t    = order["type"]
        size = order["size"]
        if t == "market_buy":
            r = market_buy(engine, size)
        elif t == "market_sell":
            r = market_sell(engine, size)
        elif t == "limit_bid":
            r = place_limit(engine, "bid", order["price"], size)
        elif t == "limit_ask":
            r = place_limit(engine, "ask", order["price"], size)
        elif t == "ioc_buy":
            r = ioc_buy(engine, size)
        elif t == "fok_buy":
            r = fok_buy(engine, size)
        else:
            r = {"error": f"tipo desconocido: {t}"}
        results.append({"order_type": t, "size": size, **r})
    return results


# Ejecutar la misma secuencia de prueba
engine_sol = MatchingEngine.from_row(row)
sol_results = simulate_sequence(engine_sol, test_orders)

for i, r in enumerate(sol_results):
    t = r["order_type"]
    if r.get("killed"):
        print(f"  {i+1}. {t:<14} KILLED — {r.get('reason', '')}")
    elif r.get("type") == "LIMIT-RESTING":
        print(f"  {i+1}. {t:<14} en cola @ ${r['price']:,.2f} x {r['size']} BTC (id={r['id']})")
    elif r.get("total_filled", 0) > 0:
        print(f"  {i+1}. {t:<14} filled={r['total_filled']:.4f} BTC  avg=${r['avg_price']:,.2f}  slip=${r['slippage']:.2f}")
    else:
        print(f"  {i+1}. {t:<14} {r}")

---

## Resumen

Has construido un `MatchingEngine` minimo que implementa los cuatro tipos fundamentales de ordenes:

| Orden | Garantia | Riesgo | Rol |
|---|---|---|---|
| Market buy/sell | Ejecucion garantizada | Precio incierto (slippage) | Taker |
| Limit bid/ask | Precio garantizado | Puede no ejecutar | Maker |
| IOC | Ejecuta lo que pueda | Residuo se pierde | Taker |
| FOK | Todo o nada | Si falta liquidez, no ejecuta nada | Taker |

**Ideas clave:**
- Una market order camina el libro nivel a nivel: el slippage crece con el tamano
- Una limit order en el spread no resting — cruza y ejecuta como market
- IOC y FOK son variantes de la market order que controlan que pasa con el residuo

## Siguiente: Lesson 6

Ahora que sabes como se generan y ejecutan ordenes, en L6 construiras un pipeline de ciencia de datos para modelizar variables del LOB.

Pregunta natural: "Esta limit order que acabo de colocar, ¿cuantas probabilidades tiene de ejecutarse?"

Esa es exactamente la variable objetivo de L6: probabilidad de fill como funcion del imbalance, el spread y la distancia al mid.